# ChatBot TODO z NLP (Slack)

Notatnik zawiera całe opracowanie chatbota. Wykonaną według załozeń 1. konserwatoriów

## 1. Biblioteki

Poza typowym importem bibliotek, przy pierwszym uruchomieniu NLTK pobiera dane potrzebne do tokenizacji.

In [1]:
import html
import os
import re
import sqlite3
import unicodedata
import uuid
from datetime import datetime

import nltk
import spacy
from dotenv import load_dotenv
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from pystempel import Stemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from slack_bolt import App
from slack_bolt.adapter.socket_mode import SocketModeHandler
from spacy.lang.pl.stop_words import STOP_WORDS

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab")

## 2. Konfiguracja

Wczytywanie kodów dostępów do Slacka zdefiniowanych w pliku środowiskowym `.env`.

In [2]:
load_dotenv()
ALBUM_NUMBER = os.getenv("ALBUM_NUMBER", "64823")

## 3. Baza danych

Zadania zapisujemy w pliku `tasks.db`. Każde zadanie ma id (UUID), treść, opcjonalną relację
do zadania nadrzędnego, status oraz daty utworzenia i modyfikacji. Kazda z operacji ma swoją osobną metodę z query do bazy danych.

In [3]:
DB_FILE = "tasks.db"


def connect():
    conn = sqlite3.connect(DB_FILE)
    conn.row_factory = sqlite3.Row 
    return conn


def init_db():
    with connect() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS tasks (
                id          TEXT PRIMARY KEY,
                description TEXT NOT NULL,
                relation    TEXT,
                status      TEXT NOT NULL,
                created_at  TEXT NOT NULL,
                updated_at  TEXT NOT NULL
            )
        """)


def now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def db_add_task(description, relation=None):
    task_id = str(uuid.uuid4())
    with connect() as conn:
        conn.execute(
            "INSERT INTO tasks VALUES (?, ?, ?, ?, ?, ?)",
            (task_id, description, relation, "todo", now(), now()),
        )
    return task_id


def db_get_all_tasks():
    with connect() as conn:
        return conn.execute("SELECT * FROM tasks ORDER BY created_at").fetchall()


def db_get_task(task_id):
    with connect() as conn:
        return conn.execute("SELECT * FROM tasks WHERE id = ?", (task_id,)).fetchone()


def db_update_description(task_id, description):
    with connect() as conn:
        conn.execute(
            "UPDATE tasks SET description = ?, updated_at = ? WHERE id = ?",
            (description, now(), task_id),
        )


def db_update_status(task_id, status):
    with connect() as conn:
        conn.execute(
            "UPDATE tasks SET status = ?, updated_at = ? WHERE id = ?",
            (status, now(), task_id),
        )


def db_delete_task(task_id):
    with connect() as conn:
        conn.execute("DELETE FROM tasks WHERE id = ?", (task_id,))
        # podzadania usuniętego zadania tracą relację
        conn.execute("UPDATE tasks SET relation = NULL WHERE relation = ?", (task_id,))


init_db()

## 4. Operacje NLP

Każdą operację najpierw definiujemy, a potem sprawdzamy na przykładowym zdaniu.
Na początku ładujemy model spaCy i stemmer Stempel.

In [4]:
nlp_pl = spacy.load("pl_core_news_sm")
stemmer = Stemmer.default()

example = "Przygotować prezentację z przetwarzania języka naturalnego"

Loading: 100%|██████████| 2225192/2225192 [00:00<00:00, 5930459.44bytes/s]


### Tokenizacja

Tokenizacja dzieli tekst na pojedyncze elementy (słowa, liczby, znaki). Używamy dwóch metod, NLTK i spaCy,
które różnią się np. obsługą skrótów i dat.

In [5]:
def tokenize_nltk(text):
    return word_tokenize(text, language="polish")


def tokenize_spacy(text):
    return [token.text for token in nlp_pl(text)]


print("NLTK: ", tokenize_nltk(example))
print("spaCy:", tokenize_spacy(example))

NLTK:  ['Przygotować', 'prezentację', 'z', 'przetwarzania', 'języka', 'naturalnego']
spaCy: ['Przygotować', 'prezentację', 'z', 'przetwarzania', 'języka', 'naturalnego']


### Czyszczenie tekstu

Funkcja `clean_text` zostawia w tekście same słowa. Usuwa liczby, interpunkcję, znaki specjalne,
pojedyncze litery i zbędne spacje.

In [6]:
def clean_text(text):
    text = text.lower()                       # małe litery
    text = re.sub(r"\d+", " ", text)          # usunięcie liczb
    text = re.sub(r"[^\w\s]|_", " ", text)    # usunięcie interpunkcji i znaków specjalnych
    text = re.sub(r"\b\w\b", " ", text)       # usunięcie pojedynczych znaków
    text = re.sub(r"\s+", " ", text)          # usunięcie nadmiarowych białych znaków
    return text.strip()


clean_text(example)

'przygotować prezentację przetwarzania języka naturalnego'

### Normalizacja tekstu

Normalizacja ujednolica zapis tekstu, ale nie usuwa słów.

In [7]:
def normalize_text(text, remove_diacritics=False):
    text = unicodedata.normalize("NFKC", text)  # ujednolicenie znaków Unicode
    text = text.replace("„", '"').replace("”", '"').replace("–", "-").replace("—", "-")
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()    # normalizacja białych znaków
    if remove_diacritics:
        text = text.replace("ł", "l")  # "ł" nie rozkłada się na literę + znak diakrytyczny
        text = unicodedata.normalize("NFKD", text)
        text = "".join(c for c in text if not unicodedata.combining(c))
    return text


print(normalize_text(example))
print(normalize_text(example, remove_diacritics=True))

przygotować prezentację z przetwarzania języka naturalnego
przygotowac prezentacje z przetwarzania jezyka naturalnego


### N-gramy

Rozmiar x-gramu zależy od ostatniej cyfry numeru albumu: cyfry 4–9 to od razu x, a do cyfr 0–3 dodajemy 4.

In [8]:
def get_ngrams(text, n):
    tokens = [t for t in tokenize_nltk(normalize_text(text)) if t.isalnum()]
    return list(ngrams(tokens, n))


def get_x(album_number):
    last_digit = int(album_number[-1])
    if last_digit >= 4:
        return last_digit
    return last_digit + 4


x = get_x(ALBUM_NUMBER)
print("Unigramy:", get_ngrams(example, 1))
print("Bigramy: ", get_ngrams(example, 2))
print("Trigramy:", get_ngrams(example, 3))
print(f"{x}-gramy: ", get_ngrams(example, x))

Unigramy: [('przygotować',), ('prezentację',), ('z',), ('przetwarzania',), ('języka',), ('naturalnego',)]
Bigramy:  [('przygotować', 'prezentację'), ('prezentację', 'z'), ('z', 'przetwarzania'), ('przetwarzania', 'języka'), ('języka', 'naturalnego')]
Trigramy: [('przygotować', 'prezentację', 'z'), ('prezentację', 'z', 'przetwarzania'), ('z', 'przetwarzania', 'języka'), ('przetwarzania', 'języka', 'naturalnego')]
7-gramy:  []


### Bag of Words

Bag of Words zlicza, ile razy każde słowo występuje w tekście. Słownik budujemy ze wszystkich zadań,
a wynik pokazujemy dla jednego z nich.

In [9]:
BOW_PARAMS = {
    "lowercase": True,
    "ngram_range": (1, 2),
    "max_features": 50,
    "min_df": 1,
    "max_df": 1.0,
    "binary": False,
}


def bag_of_words(texts, index):
    vectorizer = CountVectorizer(**BOW_PARAMS)
    matrix = vectorizer.fit_transform(texts)
    words = vectorizer.get_feature_names_out()
    row = matrix.toarray()[index]
    return {word: int(value) for word, value in zip(words, row) if value > 0}


example_tasks = [
    example,
    "Przygotować raport z laboratorium",
    "Kupić mleko i chleb",
]
bag_of_words(example_tasks, 1)

{'laboratorium': 1,
 'przygotować': 1,
 'przygotować raport': 1,
 'raport': 1,
 'raport laboratorium': 1}

### TF-IDF

TF-IDF też liczy słowa, ale obniża wagę tych, które pojawiają się w wielu zadaniach.
Dzięki temu najwyższe wagi mają słowa charakterystyczne dla danego zadania.

In [10]:
TFIDF_PARAMS = {
    "lowercase": True,
    "ngram_range": (1, 2),
    "max_features": 50,
    "min_df": 1,
    "max_df": 1.0,
    "use_idf": True,
    "smooth_idf": True,
    "sublinear_tf": True,
}


def tfidf(texts, index):
    vectorizer = TfidfVectorizer(**TFIDF_PARAMS)
    matrix = vectorizer.fit_transform(texts)
    words = vectorizer.get_feature_names_out()
    row = matrix.toarray()[index]
    return {word: round(float(value), 3) for word, value in zip(words, row) if value > 0}


tfidf(example_tasks, 1)

{'laboratorium': 0.467,
 'przygotować': 0.355,
 'przygotować raport': 0.467,
 'raport': 0.467,
 'raport laboratorium': 0.467}

### Stemming

Stemming obcina końcówki słów i zostawia ich rdzeń. NLTK nie ma stemmera dla polskiego, więc używamy algorytmu Stempel.

In [11]:
def stem(text):
    words = [t.lower() for t in tokenize_nltk(text) if t.isalpha()]
    result = []
    for word in words:
        # dla bardzo krótkich słów (np. "z", "do") Stempel zwraca bzdury, więc je pomijamy
        stemmed = stemmer(word) if len(word) > 2 else word
        result.append(stemmed or word)
    return result


stem(example)

['przygotować', 'prezentacja', 'z', 'przetwarzać', 'języka', 'naturalny']

### Lematyzacja

Lematyzacja zamienia słowo na jego formę słownikową, np. „języka” → „język”. Robi to polski model spaCy.

In [12]:
def lemmatize(text):
    return [token.lemma_ for token in nlp_pl(text) if not token.is_punct]


lemmatize(example)

['przygotować', 'prezentacja', 'z', 'przetwarzać', 'język', 'naturalny']

### Usuwanie stopwords

Stopwords to częste słowa, które niewiele wnoszą, np. „z”, „i”, „do”. Polską listę bierzemy ze spaCy, bo NLTK jej nie ma.

In [13]:
def remove_stopwords(text):
    words = [t for t in tokenize_nltk(text) if t.isalnum()]
    return " ".join(w for w in words if w.lower() not in STOP_WORDS)


print("Oryginał:     ", example)
print("Bez stopwords:", remove_stopwords(example))

Oryginał:      Przygotować prezentację z przetwarzania języka naturalnego
Bez stopwords: Przygotować prezentację przetwarzania języka naturalnego


## 5. Komendy bota

Każda komenda to zwykła funkcja, która dostaje tekst wpisany po komendzie i zwraca odpowiedź.

In [14]:
STATUSES = ["todo", "in_progress", "done"]
OPERATIONS = [
    "tokens", "clean", "normalize", "unigram", "bigram", "trigram", "xgram",
    "bow", "tfidf", "stem", "lemma", "stopwords",
]


def tasks_command():
    tasks = db_get_all_tasks()
    if not tasks:
        return "Brak zadań."
    lines = []
    for task in tasks:
        line = f"• {task['description']}\n    id: `{task['id']}` | status: {task['status']}"
        if task["relation"]:
            line += f" | zadanie nadrzędne: `{task['relation']}`"
        lines.append(line)
    return "\n".join(lines)


def add_command(text):
    description, _, relation = text.partition("|")
    description = description.strip()
    relation = relation.strip() or None
    if not description:
        return "Użycie: /add [treść zadania] | [id zadania nadrzędnego - opcjonalnie]"
    if relation and db_get_task(relation) is None:
        return "Nie ma zadania nadrzędnego o takim id."
    task_id = db_add_task(description, relation)
    return f"Utworzono zadanie: `{task_id}`"


def update_command(text):
    parts = text.split(maxsplit=1)
    if len(parts) < 2:
        return "Użycie: /update [id] [nowa treść] albo /update [id] status=done"
    task_id, new_value = parts
    if db_get_task(task_id) is None:
        return "Nie znaleziono zadania o takim id."

    if new_value.startswith("status="):
        status = new_value.removeprefix("status=").strip()
        if status not in STATUSES:
            return "Dozwolone statusy: " + ", ".join(STATUSES)
        db_update_status(task_id, status)
    else:
        db_update_description(task_id, new_value)
    return f"Zaktualizowano zadanie `{task_id}`"


def delete_command(text):
    task_id = text.strip()
    if db_get_task(task_id) is None:
        return "Nie znaleziono zadania o takim id."
    db_delete_task(task_id)
    return f"Usunięto zadanie `{task_id}`"


def task_command(text):
    parts = text.split()
    if len(parts) != 2:
        return "Użycie: /task [id] [operacja]\nOperacje: " + ", ".join(OPERATIONS)
    task_id, operation = parts
    task = db_get_task(task_id)
    if task is None:
        return "Nie znaleziono zadania o takim id."

    description = task["description"]
    all_tasks = db_get_all_tasks()
    texts = [t["description"] for t in all_tasks]
    index = [t["id"] for t in all_tasks].index(task_id)

    if operation == "tokens":
        nltk_tokens = tokenize_nltk(description)
        spacy_tokens = tokenize_spacy(description)
        same = "tak" if nltk_tokens == spacy_tokens else "nie"
        result = f"NLTK:  {nltk_tokens}\nspaCy: {spacy_tokens}\nWyniki identyczne: {same}"
    elif operation == "clean":
        result = clean_text(description)
    elif operation == "normalize":
        result = (
            f"{normalize_text(description)}\n"
            f"Bez znaków diakrytycznych: {normalize_text(description, remove_diacritics=True)}"
        )
    elif operation == "unigram":
        result = get_ngrams(description, 1)
    elif operation == "bigram":
        result = get_ngrams(description, 2)
    elif operation == "trigram":
        result = get_ngrams(description, 3)
    elif operation == "xgram":
        x = get_x(ALBUM_NUMBER)
        result = f"x = {x}\n{get_ngrams(description, x)}"
    elif operation == "bow":
        result = f"Parametry: {BOW_PARAMS}\n{bag_of_words(texts, index)}"
    elif operation == "tfidf":
        result = f"Parametry: {TFIDF_PARAMS}\n{tfidf(texts, index)}"
    elif operation == "stem":
        result = stem(description)
    elif operation == "lemma":
        result = lemmatize(description)
    elif operation == "stopwords":
        result = f"Tekst oryginalny: {description}\nBez stopwords:    {remove_stopwords(description)}"
    else:
        return "Nieznana operacja. Dostępne: " + ", ".join(OPERATIONS)

    return f"```{result}```"

### Test komend

Dodajemy przykładowe zadanie, sprawdzamy na nim kilka komend, a na końcu je usuwamy.

In [15]:
answer = add_command("Przygotować prezentację z przetwarzania języka naturalnego")
print(answer)
test_id = answer.split("`")[1]

print(tasks_command())
print(task_command(f"{test_id} tokens"))
print(task_command(f"{test_id} bigram"))
print(update_command(f"{test_id} status=done"))
print(delete_command(test_id))

Utworzono zadanie: `268acc96-9c83-4241-b50d-f16c2a1ab8ef`
• Przygotować prezentację z przetwarzania języka naturalnego
    id: `e6c85c6c-9604-4451-84af-1f091e399b90` | status: todo
• Przygotować prezentację z przetwarzania języka naturalnego
    id: `ed665d65-d36a-44ed-9870-fa3636adeb57` | status: todo
• Przygotować prezentację z przetwarzania języka naturalnego
    id: `268acc96-9c83-4241-b50d-f16c2a1ab8ef` | status: todo
```NLTK:  ['Przygotować', 'prezentację', 'z', 'przetwarzania', 'języka', 'naturalnego']
spaCy: ['Przygotować', 'prezentację', 'z', 'przetwarzania', 'języka', 'naturalnego']
Wyniki identyczne: tak```
```[('przygotować', 'prezentację'), ('prezentację', 'z'), ('z', 'przetwarzania'), ('przetwarzania', 'języka'), ('języka', 'naturalnego')]```
Zaktualizowano zadanie `268acc96-9c83-4241-b50d-f16c2a1ab8ef`
Usunięto zadanie `268acc96-9c83-4241-b50d-f16c2a1ab8ef`


## 6. Uruchomienie bota w Slacku

Łączymy komendy ze Slackiem i uruchamiamy bota. Bot działa w tle, dopóki notatnik jest otwarty.

In [16]:
app = App(token=os.environ["SLACK_BOT_TOKEN"])


def get_text(command):
    return html.unescape(command.get("text", ""))


def reply(respond, text):
    respond(text=text, response_type="in_channel") 


@app.command("/tasks")
def handle_tasks(ack, respond):
    ack() 
    reply(respond, tasks_command())


@app.command("/add")
def handle_add(ack, respond, command):
    ack()
    reply(respond, add_command(get_text(command)))


@app.command("/update")
def handle_update(ack, respond, command):
    ack()
    reply(respond, update_command(get_text(command)))


@app.command("/delete")
def handle_delete(ack, respond, command):
    ack()
    reply(respond, delete_command(get_text(command)))


@app.command("/task")
def handle_task(ack, respond, command):
    ack()
    reply(respond, task_command(get_text(command)))

COMMANDS = {
    "tasks": lambda text: tasks_command(),
    "add": add_command,
    "update": update_command,
    "delete": delete_command,
    "task": task_command,
}

@app.event("message")
def handle_message(event, say):
    if event.get("channel_type") != "im" or event.get("bot_id") or event.get("subtype"):
        return
    text = html.unescape(event.get("text", "")).strip().lstrip("/")
    name, _, rest = text.partition(" ")
    if name in COMMANDS:
        say(COMMANDS[name](rest))
    else:
        say("Nieznana komenda. Dostępne: " + ", ".join(COMMANDS))


handler = SocketModeHandler(app, os.environ["SLACK_APP_TOKEN"])
handler.connect()
print("Bot działa!")

Bot działa!


Uruchom tę komórkę, aby zatrzymać bota.

In [17]:
# handler.close()
# print("Bot zatrzymany.")